**Koneksi ke MongoDB**

In [1]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"] # database baru
menu = db["menu"] # collection baru

print("Koneksi ke MongoDB berhasil!")
print("Daftar database yang ada:", client.list_database_names())

Koneksi ke MongoDB berhasil!
Daftar database yang ada: ['admin', 'config', 'local', 'praktikum_bigdata']


**A. Membangun Katalog (Create)**

Database kopi_nusantara, collection menu, berisi 14 dokumen. Field-nya sengaja tidak seragam:
- Kopi punya tingkat_kepahitan (1-5) dan rekomendasi_barista
- Makanan punya pedas_level (0-3)
- Minuman Non-Kopi punya tingkat_manis (1-5)
- Semua dokumen punya nama_menu, harga, kategori, stok, dan tags

In [2]:
# Hapus data lama dulu supaya notebook aman di-Run All berkali-kali
menu.delete_many({})

data_menu = [
    # ---- Kategori Kopi (field khas: tingkat_kepahitan, rekomendasi_barista) ----
    {"nama_menu": "Espresso",            "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 5, "rekomendasi_barista": True,  "stok": 50, "tags": ["strong", "hitam"]},
    {"nama_menu": "Americano",           "harga": 22000, "kategori": "Kopi", "tingkat_kepahitan": 4, "rekomendasi_barista": False, "stok": 60, "tags": ["hitam"]},
    {"nama_menu": "Kopi Tubruk Robusta", "harga": 15000, "kategori": "Kopi", "tingkat_kepahitan": 5, "rekomendasi_barista": True,  "stok": 40, "tags": ["tradisional", "strong"]},
    {"nama_menu": "V60 Arabika Gayo",    "harga": 32000, "kategori": "Kopi", "tingkat_kepahitan": 3, "rekomendasi_barista": True,  "stok": 25, "tags": ["manual-brew", "arabika"]},
    {"nama_menu": "Cappuccino",          "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 3, "rekomendasi_barista": False, "stok": 70, "tags": ["susu", "hangat"]},
    {"nama_menu": "Kopi Susu Gula Aren", "harga": 20000, "kategori": "Kopi", "tingkat_kepahitan": 2, "rekomendasi_barista": False, "stok": 90, "tags": ["best-seller", "dingin"]},
    {"nama_menu": "Caramel Latte",       "harga": 28000, "kategori": "Kopi", "tingkat_kepahitan": 1, "rekomendasi_barista": False, "stok": 55, "tags": ["manis", "dingin"]},

    # ---- Kategori Makanan (field khas: pedas_level) ----
    {"nama_menu": "Roti Bakar Cokelat",  "harga": 15000, "kategori": "Makanan", "pedas_level": 0, "stok": 45, "tags": ["manis", "hangat"]},
    {"nama_menu": "Kentang Goreng",      "harga": 20000, "kategori": "Makanan", "pedas_level": 1, "stok": 60, "tags": ["gurih"]},
    {"nama_menu": "Mie Goreng Pedas",    "harga": 27000, "kategori": "Makanan", "pedas_level": 3, "stok": 30, "tags": ["pedas", "best-seller"]},
    {"nama_menu": "Nasi Goreng Kampung", "harga": 30000, "kategori": "Makanan", "pedas_level": 2, "stok": 35, "tags": ["nasi", "gurih"]},
    {"nama_menu": "Sandwich Ayam",       "harga": 24000, "kategori": "Makanan", "pedas_level": 0, "stok": 28, "tags": ["ayam"]},

    # ---- Kategori Minuman Non-Kopi (field khas: tingkat_manis) ----
    {"nama_menu": "Es Teh Lemon",        "harga": 12000, "kategori": "Minuman Non-Kopi", "tingkat_manis": 3, "stok": 100, "tags": ["dingin", "segar"]},
    {"nama_menu": "Matcha Latte",        "harga": 26000, "kategori": "Minuman Non-Kopi", "tingkat_manis": 4, "stok": 40,  "tags": ["susu", "dingin"]},
]

hasil = menu.insert_many(data_menu)
print("Jumlah dokumen yang disisipkan:", len(hasil.inserted_ids))
print("Total dokumen di collection 'menu':", menu.count_documents({}))

print("\nJumlah dokumen per kategori:")
for r in menu.aggregate([{"$group": {"_id": "$kategori", "jumlah": {"$sum": 1}}}, {"$sort": {"_id": 1}}]):
    print(f"  - {r['_id']}: {r['jumlah']}")

Jumlah dokumen yang disisipkan: 14
Total dokumen di collection 'menu': 14

Jumlah dokumen per kategori:
  - Kopi: 7
  - Makanan: 5
  - Minuman Non-Kopi: 2


**B. Kueri Bertingkat (Read)**

In [3]:
# Kueri 1: menu kategori "Kopi" dengan tingkat_kepahitan di atas 3 (> 3)
print("1) Menu Kopi dengan tingkat_kepahitan > 3:")
for p in menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}, {"_id": 0, "nama_menu": 1, "harga": 1, "tingkat_kepahitan": 1}):
    print(f"  - {p['nama_menu']} | kepahitan = {p['tingkat_kepahitan']} | Rp{p['harga']:,}")

1) Menu Kopi dengan tingkat_kepahitan > 3:
  - Espresso | kepahitan = 5 | Rp18,000
  - Americano | kepahitan = 4 | Rp22,000
  - Kopi Tubruk Robusta | kepahitan = 5 | Rp15,000


In [4]:
# Kueri 2: menu dengan harga di antara 15000 dan 30000 (batas ikut dihitung: >= 15000 dan <= 30000)
print("2) Menu dengan harga 15.000 - 30.000:")
for p in menu.find({"harga": {"$gte": 15000, "$lte": 30000}}, {"_id": 0, "nama_menu": 1, "kategori": 1, "harga": 1}).sort("harga", 1):
    print(f"  - {p['nama_menu']} ({p['kategori']}) - Rp{p['harga']:,}")

2) Menu dengan harga 15.000 - 30.000:
  - Kopi Tubruk Robusta (Kopi) - Rp15,000
  - Roti Bakar Cokelat (Makanan) - Rp15,000
  - Espresso (Kopi) - Rp18,000
  - Kopi Susu Gula Aren (Kopi) - Rp20,000
  - Kentang Goreng (Makanan) - Rp20,000
  - Americano (Kopi) - Rp22,000
  - Sandwich Ayam (Makanan) - Rp24,000
  - Cappuccino (Kopi) - Rp25,000
  - Matcha Latte (Minuman Non-Kopi) - Rp26,000
  - Mie Goreng Pedas (Makanan) - Rp27,000
  - Caramel Latte (Kopi) - Rp28,000
  - Nasi Goreng Kampung (Makanan) - Rp30,000


In [5]:
# Kueri 3: menu yang TIDAK memiliki field pedas_level sama sekali ($exists: False)
print("3) Menu tanpa field 'pedas_level':")
for p in menu.find({"pedas_level": {"$exists": False}}, {"_id": 0, "nama_menu": 1, "kategori": 1}):
    print(f"  - {p['nama_menu']} ({p['kategori']})")

3) Menu tanpa field 'pedas_level':
  - Espresso (Kopi)
  - Americano (Kopi)
  - Kopi Tubruk Robusta (Kopi)
  - V60 Arabika Gayo (Kopi)
  - Cappuccino (Kopi)
  - Kopi Susu Gula Aren (Kopi)
  - Caramel Latte (Kopi)
  - Es Teh Lemon (Minuman Non-Kopi)
  - Matcha Latte (Minuman Non-Kopi)


**C. Pembaruan Data (Update)**

In [6]:
# 1) Tambahkan field status = "Tersedia" ke SELURUH dokumen
r1 = menu.update_many({}, {"$set": {"status": "Tersedia"}})
print("update_many -> cocok:", r1.matched_count, "| diubah:", r1.modified_count)

# 2) Ubah status jadi "Habis" untuk 2 menu pilihan, masing-masing dengan update_one()
r2 = menu.update_one({"nama_menu": "Kopi Tubruk Robusta"}, {"$set": {"status": "Habis"}})
r3 = menu.update_one({"nama_menu": "Mie Goreng Pedas"},    {"$set": {"status": "Habis"}})
print("update_one (Kopi Tubruk Robusta) -> diubah:", r2.modified_count)
print("update_one (Mie Goreng Pedas)    -> diubah:", r3.modified_count)

print("\nStatus seluruh menu setelah update:")
for p in menu.find({}, {"_id": 0, "nama_menu": 1, "status": 1}):
    print(f"  - {p['nama_menu']}: {p['status']}")

print("\nJumlah per status:")
for r in menu.aggregate([{"$group": {"_id": "$status", "jumlah": {"$sum": 1}}}]):
    print(f"  - {r['_id']}: {r['jumlah']}")

update_many -> cocok: 14 | diubah: 14
update_one (Kopi Tubruk Robusta) -> diubah: 1
update_one (Mie Goreng Pedas)    -> diubah: 1

Status seluruh menu setelah update:
  - Espresso: Tersedia
  - Americano: Tersedia
  - Kopi Tubruk Robusta: Habis
  - V60 Arabika Gayo: Tersedia
  - Cappuccino: Tersedia
  - Kopi Susu Gula Aren: Tersedia
  - Caramel Latte: Tersedia
  - Roti Bakar Cokelat: Tersedia
  - Kentang Goreng: Tersedia
  - Mie Goreng Pedas: Habis
  - Nasi Goreng Kampung: Tersedia
  - Sandwich Ayam: Tersedia
  - Es Teh Lemon: Tersedia
  - Matcha Latte: Tersedia

Jumlah per status:
  - Tersedia: 12
  - Habis: 2


**D. Integrasi ke Spark**

In [7]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count, round as sround

spark = SparkSession.builder \
    .appName("Tugas9-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Langkah 1: ambil seluruh dokumen dari MongoDB
dokumen_mongo = list(menu.find({}))

# Langkah 2: ubah jadi pandas DataFrame
df_pandas = pd.DataFrame(dokumen_mongo)

# Langkah 3: sesuaikan tipe khusus MongoDB
df_pandas["_id"] = df_pandas["_id"].astype(str)
df_pandas["tags"] = df_pandas["tags"].apply(lambda x: ", ".join(x) if isinstance(x, list) else "")

# Langkah 4 (tambahan): field yang tidak ada di sebagian dokumen jadi NaN di pandas.
# Ubah NaN -> None supaya Spark membacanya sebagai null
df_pandas = df_pandas.astype(object).where(df_pandas.notna(), None)

# Langkah 5: konversi ke Spark DataFrame
df_spark_menu = spark.createDataFrame(df_pandas)
df_spark_menu.show(truncate=False)

26/10/29 14:26:46 WARN Utils: Your hostname, xcel resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/10/29 14:26:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/29 14:26:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

+------------------------+-------------------+-----+----------------+-----------------+-------------------+----+--------------------+--------+-----------+-------------+
|_id                     |nama_menu          |harga|kategori        |tingkat_kepahitan|rekomendasi_barista|stok|tags                |status  |pedas_level|tingkat_manis|
+------------------------+-------------------+-----+----------------+-----------------+-------------------+----+--------------------+--------+-----------+-------------+
|6ae2f033a24a71921a06260d|Espresso           |18000|Kopi            |5.0              |true               |50  |strong, hitam       |Tersedia|NULL       |NULL         |
|6ae2f033a24a71921a06260e|Americano          |22000|Kopi            |4.0              |false              |60  |hitam               |Tersedia|NULL       |NULL         |
|6ae2f033a24a71921a06260f|Kopi Tubruk Robusta|15000|Kopi            |5.0              |true               |40  |tradisional, strong |Habis   |NULL       |N

In [8]:
df_spark_menu.printSchema()

root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- tingkat_kepahitan: double (nullable = true)
 |-- rekomendasi_barista: boolean (nullable = true)
 |-- stok: long (nullable = true)
 |-- tags: string (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: double (nullable = true)
 |-- tingkat_manis: double (nullable = true)



In [9]:
df_spark_menu.groupBy("kategori").agg(
    sround(avg("harga"), 2).alias("rata_rata_harga"),
    count("nama_menu").alias("jumlah_menu")
).orderBy("kategori").show()

spark.stop()
print("SparkSession ditutup.")

+----------------+---------------+-----------+
|        kategori|rata_rata_harga|jumlah_menu|
+----------------+---------------+-----------+
|            Kopi|       22857.14|          7|
|         Makanan|        23200.0|          5|
|Minuman Non-Kopi|        19000.0|          2|
+----------------+---------------+-----------+

SparkSession ditutup.


**E. Kesimpulan**

Tantangan utamanya ada pada perbedaan cara MongoDB dan Spark memandang skema. Di MongoDB, setiap dokumen boleh punya field yang berbeda: hanya menu Kopi yang punya *tingkat_kepahitan* dan *rekomendasi_barista*, hanya Makanan yang punya *pedas_level*, dan hanya Minuman Non-Kopi yang punya *tingkat_manis*. Sementara itu, Spark membutuhkan kolom yang seragam di semua baris. Saat aku mencoba pola sub-bab 6.10 apa adanya, pandas mengisi field yang kosong dengan *NaN*, sehingga kolom *rekomendasi_barista* berisi True/False bercampur *NaN* dan Spark menampilkan error *CANNOT_MERGE_TYPE*. Kolom angka lainnya juga berubah menjadi *double* (misalnya 5.0). Aku mengatasinya dengan mengubah *NaN* menjadi *None* lewat **df_pandas.astype(object).where(df_pandas.notna(), None)** sebelum **createDataFrame**, sehingga field yang kosong terbaca sebagai *null* di Spark. Setelah itu konversi berhasil dan **groupBy** bisa dipakai tanpa masalah.